# Stage 3B.01 — setup and preflight

In [ ]:
import json,os,subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; PY=Path.home()/"venv-stage1-id/bin/python"; S3=Path.home()/"stage3"; OUT=Path.home()/"stage3b"; OUT.mkdir(exist_ok=True)
for p in (R,PY,S3/"stage3_manifest.csv",S3/"stage3_episode_results.csv"): 
    if not p.exists(): raise SystemExit(f"STOP: missing {p}")
env=os.environ.copy(); env.update({"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg"})
subprocess.run([str(PY),"-m","pytest","-q",str(R/"async_vla_benchmark/tests")],cwd=R,env=env,check=True)
gpu="1"; line=subprocess.run(["nvidia-smi",f"--id={gpu}","--query-gpu=name,memory.total,memory.used,utilization.gpu,driver_version","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True).stdout.strip(); print(line)
name,total,used,util,driver=[x.strip() for x in line.split(',')]; assert "A100" in name
if int(used)>=500 or int(util)>=5: raise SystemExit("STOP: select an idle A100")
provenance={"repository_sha":subprocess.run(["git","-C",str(R),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip(),"gpu":line,"stage3_results":str(S3/"stage3_episode_results.csv")}
(OUT/"stage3b_preflight_environment.json").write_text(json.dumps(provenance,indent=2)+"\n"); print("PASS: Stage 3B preflight complete")